In [1]:
!pip install optuna mlflow dagshub seaborn

In [2]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import dagshub
import mlflow
import mlflow.sklearn

In [3]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [4]:
df = pd.read_csv("/kaggle/input/datasets/aryanumri/twitter-sentiment/twitter_cleaned (2).csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [5]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

# Prevent CV indexing errors
y_train = np.array(y_train)
y_test = np.array(y_test)

In [6]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (41292, 8000)
Test shape: (10323, 8000)


In [7]:
def build_linsvc(trial):

    C = trial.suggest_float("C", 1e-4, 10.0, log=True)
    loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])

    return LinearSVC(
        C=C,
        loss=loss,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    )

In [8]:
def objective(trial):

    model = build_linsvc(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(trial.params)
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [9]:
study = optuna.create_study(
    direction="maximize",
    study_name="linear_svc_study",
    storage="sqlite:///linear_svc_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 09:19:16,272] A new study created in RDB with name: linear_svc_study


In [10]:
with mlflow.start_run(run_name="LinearSVC"):

    mlflow.log_param("model_name", "LinearSVC")
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 8000)
    mlflow.log_param("min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = LinearSVC(
        **best_params,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    )

    final_model.fit(X_train, y_train)

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = LinearSVC(
            **best_params,
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_linsvc.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_linsvc.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - LinearSVC")
    plt.savefig("confusion_matrix_linsvc.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_linsvc.png")

    study.trials_dataframe().to_csv("optuna_trials_linsvc.csv", index=False)
    mlflow.log_artifact("optuna_trials_linsvc.csv")

    mlflow.sklearn.log_model(final_model, artifact_path="model")

print("LinearSVC experiment completed successfully.")

🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/bffba0b5335c40b293769f970251dc1e
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:19:20,577] Trial 0 finished with value: 0.6206148232112526 and parameters: {'C': 0.00016583076013131505, 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.6206148232112526.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/4b976d1cd72f48c1b9a2055e196655d8
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:19:27,591] Trial 1 finished with value: 0.6895806390499398 and parameters: {'C': 0.008059433203657105, 'loss': 'hinge'}. Best is trial 1 with value: 0.6895806390499398.


🏃 View run trial_2 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/51f3f60aef9d412abc315ab178498d4a
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


...

🏃 View run trial_49 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/aca46bd2a2b14c6cbc2f91a74772df08
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:34:23,373] Trial 49 finished with value: 0.7034883879267944 and parameters: {'C': 0.0029462242919408746, 'loss': 'squared_hinge'}. Best is trial 46 with value: 0.8087346443803786.


Best Macro F1: 0.8087346443803786
Best Params: {'C': 0.6884207332173677, 'loss': 'squared_hinge'}
Fold 1 - Macro F1: 0.8002
Fold 2 - Macro F1: 0.7989
Fold 3 - Macro F1: 0.7942
Fold 4 - Macro F1: 0.8027
Fold 5 - Macro F1: 0.7955
Final Test Macro F1: 0.8087346443803786


2026/02/23 09:35:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 09:35:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearSVC at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/b40139fa6d534f9aab8f5c3e5660f43d
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
LinearSVC experiment completed successfully.
